In [ ]:
# -- Dataset Path Configuration ------------------------------------------
# Set the variables below to point to your local copies of the dataset
# files. All other cells read paths from these variables.
# See README.md for instructions on obtaining the dataset.
# -------------------------------------------------------------------------

# Parquet files -- used by the final training pipeline (Cells 3, 4, 8, 9)
X_PARQUET_PATH        = 'path/to/x_features.parquet'   # 37,535 x 11,379
Y_PARQUET_PATH        = 'path/to/y_labels.parquet'      # 37,535 x 7

# CSV files -- used by the earlier batch pipeline (Cells 12-15, 19, 27)
TRAINING_LABEL_CSV    = 'path/to/training_label.csv'
GEE_FEATURES_CSV      = 'path/to/gee_features.csv'
TRAIN_FEATURES_CSV    = 'path/to/train_features.csv'
SAMPLE_SUBMISSION_CSV = 'path/to/sample_submission.csv'
SAMPLE_FEATURES_CSV   = 'path/to/sample_features.csv'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import sklearn
from sklearn import model_selection
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import xgboost as xgb
from xgboost import plot_importance, plot_tree
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.decomposition import PCA
from numba import jit
from csv import writer
from google.colab import files

In [ ]:
X = pd.read_parquet(X_PARQUET_PATH)
X.shape

(37535, 11379)

In [ ]:
Y = pd.read_parquet(Y_PARQUET_PATH)
Y.columns

Index(['DHSID', 'Mean_BMI', 'Median_BMI', 'Unmet_Need_Rate',
       'Under5_Mortality_Rate', 'Skilled_Birth_Attendant_Rate',
       'Stunted_Rate'],
      dtype='object')

In [ ]:
count = 0
for i in reversed(range(0,X.shape[1])):
  if(X[X.columns[i]].dtype == 'O'):
    count = count+1
    print(X.columns[i])
print(count)

DHSID
1


In [ ]:
X = pd.read_parquet(X_PARQUET_PATH)
Y = pd.read_parquet(Y_PARQUET_PATH)
# training_data = pd.read_csv(TRAINING_LABEL_CSV)
# connector = pd.read_csv(TRAIN_FEATURES_CSV)
training_data = Y[["DHSID", "Mean_BMI"]]

reg = xgb.XGBRegressor(n_estimators=2000)
# training_data = training_data.dropna(axis=0, how='any')
# training_data.reset_index(drop = True, inplace = True)
c=0
# X_testing = np.empty([0,500], int)
# Y_testing = np.empty([0,1], int)

# name_list = []
# for i in range(0,connector.shape[0]):
#   if(connector['identifier'][i] == 1):
#     name_list.append(connector['id_names'][i])

# arr = connector['identifier'].to_numpy()
# p = training_data.shape[0]

train_features = X.copy()
# train_id = train_features['DHSID'].tolist()
# val_list = []
# train_features,c,ct = long_calc(df,training_data,c)

# for i in range(0,train_features.shape[0]):
#   if(train_features['URBAN_RURA'][i] == 'R'):
#     train_features['URBAN_RURA'][i] = 0
#   else:
#     train_features['URBAN_RURA'][i] = 1
# train_features['URBAN_RURA'] = train_features['URBAN_RURA'].astype(int)

  # count = 0
  # for i in reversed(range(0,train_features.shape[1])):
  #   if(train_features[train_features.columns[i]].dtype == 'O'):
  #     count = count+1
feature_list = train_features['DHSID'].tolist()
train_features.drop(['DHSID'],axis = 1,inplace = True)
# train_features.drop(['key1','SOURCE','DHSREGNA','DHSID','DHSCC','DATUM','CCFIPS','ADM1SALBNA','ADM1SALBCO','ADM1NAME','ADM1FIPSNA','ADM1FIPS','new_ind','ZONENA'],axis = 1,inplace = True)

for i in range(train_features.shape[1]):
  mean_value=train_features[train_features.columns[i]].mean()
  train_features[train_features.columns[i]].fillna(value=mean_value, inplace=True)

for i in range(train_features.shape[1]):
  mean_value=train_features[train_features.columns[i]].mean()
  train_features[train_features.columns[i]].fillna(value=0, inplace=True)

  # arr_passed = arr[ct:(ct+batch_size)]
  # m=0
  # for i in range(0,arr_passed.shape[0]):
  #   if (arr_passed[i]==0):
  #     m = m + 1
  # print(m,arr_passed.shape[0])

# train_list = training_data['DHSID'].tolist()
# for k in range(0,train_features.shape[0]):
#   if (c < training_data.shape[0] and feature_list[k] in name_list):
#     if (train_list[c] in name_list):
#       val_list.append(training_data['Mean_BMI'][c])
#     else:
#       arr[k] = 0
#   c = c + 1
Y1 = training_data['Mean_BMI']

# train_f = train_features.to_numpy()

# for i in reversed(range(0,train_f.shape[0])):
#   if (arr[i]==0):
#     train_f = np.delete(train_f,i,0)

# if (c>p):
#   train_f = train_f[0:(p)]
# print(c,train_f.shape[0],Y.shape[0])
#   # ct = ct + batch_size

X_train, X_test, Y_train, Y_test = model_selection.train_test_split(train_features, Y1, test_size=0.3, train_size=0.7)

pca = PCA(n_components = 500)
X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)
  # explained_variance = pca.explained_variance_ratio_

reg.fit(X_train, Y_train, eval_set=[(X_train, Y_train), (X_test, Y_test)], early_stopping_rounds=750, verbose=True)

  # X_testing = np.append(X_testing, X_test, axis=0)
  # Y_testing = np.append(Y_testing, Y_test, axis=0)
Y_pred = reg.predict(X_test)
Y_prediction = pd.DataFrame(Y_pred, columns = ['Prediction'])
print(mean_squared_error(y_true = Y_test, y_pred = Y_prediction))

# if(c>p):
#   break

# Y_pred = reg.predict(X_testing)
# Y_prediction = pd.DataFrame(Y_pred, columns = ['Prediction'])
# mean_squared_error(y_true = Y_testing, y_pred = Y_prediction)

/usr/local/lib/python3.10/dist-packages/xgboost/sklearn.py:835: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


[0]	validation_0-rmse:16.69032	validation_1-rmse:16.72919
[1]	validation_0-rmse:11.84767	validation_1-rmse:11.88294
[2]	validation_0-rmse:8.50848	validation_1-rmse:8.54263
[3]	validation_0-rmse:6.24137	validation_1-rmse:6.28329
[4]	validation_0-rmse:4.72972	validation_1-rmse:4.77803
[5]	validation_0-rmse:3.76024	validation_1-rmse:3.82909
[6]	validation_0-rmse:3.16195	validation_1-rmse:3.25220
[7]	validation_0-rmse:2.80600	validation_1-rmse:2.92099
[8]	validation_0-rmse:2.59317	validation_1-rmse:2.73543
[9]	validation_0-rmse:2.47321	validation_1-rmse:2.64246
[10]	validation_0-rmse:2.39360	validation_1-rmse:2.58588
[11]	validation_0-rmse:2.33831	validation_1-rmse:2.55413
[12]	validation_0-rmse:2.31024	validation_1-rmse:2.53720
[13]	validation_0-rmse:2.27641	validation_1-rmse:2.52631
[14]	validation_0-rmse:2.25368	validation_1-rmse:2.51732
[15]	validation_0-rmse:2.22680	validation_1-rmse:2.51066
[16]	validation_0-rmse:2.20114	validation_1-rmse:2.50553
[17]	validation_0-rmse:2.17805	valida

In [ ]:
import pandas as pd
import sklearn
from sklearn import model_selection
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import xgboost as xgb
from xgboost import plot_importance, plot_tree
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.decomposition import PCA
from numba import jit
from csv import writer
from google.colab import files

X = pd.read_parquet(X_PARQUET_PATH)
Y = pd.read_parquet(Y_PARQUET_PATH)
training_data = Y[['DHSID','Mean_BMI']]
te=0
train_features = X.copy()
feature_list = train_features['DHSID'].tolist()
train_features.drop(['DHSID'],axis = 1,inplace = True)
for i in range(train_features.shape[1]):
  mean_value=train_features[train_features.columns[i]].mean()
  train_features[train_features.columns[i]].fillna(value=mean_value, inplace=True)
for i in range(train_features.shape[1]):
  mean_value=train_features[train_features.columns[i]].mean()
  train_features[train_features.columns[i]].fillna(value=0, inplace=True)

reg1 = xgb.XGBRegressor(n_estimators=2000)
Y1 = training_data['Mean_BMI']
X_train, X_test, Y_train, Y_test = model_selection.train_test_split(train_features, Y1, test_size=0.25, train_size=0.75)
pca = PCA(n_components = 500)
X_train = pca.fit_transform(X_train)
X_test = pca.fit_transform(X_test)
reg1.fit(X_train, Y_train, eval_set=[(X_train, Y_train), (X_test, Y_test)], early_stopping_rounds=500, verbose=True)
Y_pred = reg1.predict(X_test)
Y_prediction = pd.DataFrame(Y_pred, columns = ['Prediction'])
print(mean_squared_error(y_true = Y_test, y_pred = Y_prediction))
te = te + mean_squared_error(y_true = Y_test, y_pred = Y_prediction)

reg2 = xgb.XGBRegressor(n_estimators=2000)
Y2 = Y['Median_BMI']
X_train, X_test, Y_train, Y_test = model_selection.train_test_split(train_features, Y2, test_size=0.25, train_size=0.75)
pca = PCA(n_components = 500)
X_train = pca.fit_transform(X_train)
X_test = pca.fit_transform(X_test)
reg2.fit(X_train, Y_train, eval_set=[(X_train, Y_train), (X_test, Y_test)], early_stopping_rounds=500, verbose=True)
Y_pred = reg2.predict(X_test)
Y_prediction = pd.DataFrame(Y_pred, columns = ['Prediction'])
print(mean_squared_error(y_true = Y_test, y_pred = Y_prediction))
te = te + mean_squared_error(y_true = Y_test, y_pred = Y_prediction)

reg3 = xgb.XGBRegressor(n_estimators=2000)
Y3 = Y['Unmet_Need_Rate']
X_train, X_test, Y_train, Y_test = model_selection.train_test_split(train_features, Y3, test_size=0.25, train_size=0.75)
pca = PCA(n_components = 500)
X_train = pca.fit_transform(X_train)
X_test = pca.fit_transform(X_test)
reg3.fit(X_train, Y_train, eval_set=[(X_train, Y_train), (X_test, Y_test)], early_stopping_rounds=500, verbose=True)
Y_pred = reg3.predict(X_test)
Y_prediction = pd.DataFrame(Y_pred, columns = ['Prediction'])
print(mean_squared_error(y_true = Y_test, y_pred = Y_prediction))
te = te + mean_squared_error(y_true = Y_test, y_pred = Y_prediction)

reg4 = xgb.XGBRegressor(n_estimators=2000)
Y4 = Y['Under5_Mortality_Rate']
X_train, X_test, Y_train, Y_test = model_selection.train_test_split(train_features, Y4, test_size=0.25, train_size=0.75)
pca = PCA(n_components = 500)
X_train = pca.fit_transform(X_train)
X_test = pca.fit_transform(X_test)
reg4.fit(X_train, Y_train, eval_set=[(X_train, Y_train), (X_test, Y_test)], early_stopping_rounds=500, verbose=True)
Y_pred = reg4.predict(X_test)
Y_prediction = pd.DataFrame(Y_pred, columns = ['Prediction'])
print(mean_squared_error(y_true = Y_test, y_pred = Y_prediction))
te = te + mean_squared_error(y_true = Y_test, y_pred = Y_prediction)

reg5 = xgb.XGBRegressor(n_estimators=2000)
Y5 = Y['Skilled_Birth_Attendant_Rate']
X_train, X_test, Y_train, Y_test = model_selection.train_test_split(train_features, Y5, test_size=0.25, train_size=0.75)
pca = PCA(n_components = 500)
X_train = pca.fit_transform(X_train)
X_test = pca.fit_transform(X_test)
reg5.fit(X_train, Y_train, eval_set=[(X_train, Y_train), (X_test, Y_test)], early_stopping_rounds=500, verbose=True)
Y_pred = reg5.predict(X_test)
Y_prediction = pd.DataFrame(Y_pred, columns = ['Prediction'])
print(mean_squared_error(y_true = Y_test, y_pred = Y_prediction))
te = te + mean_squared_error(y_true = Y_test, y_pred = Y_prediction)

reg6 = xgb.XGBRegressor(n_estimators=2000)
Y6 = Y['Stunted_Rate']
X_train, X_test, Y_train, Y_test = model_selection.train_test_split(train_features, Y6, test_size=0.25, train_size=0.75)
pca = PCA(n_components = 500)
X_train = pca.fit_transform(X_train)
X_test = pca.fit_transform(X_test)
reg6.fit(X_train, Y_train, eval_set=[(X_train, Y_train), (X_test, Y_test)], early_stopping_rounds=500, verbose=True)
Y_pred = reg6.predict(X_test)
Y_prediction = pd.DataFrame(Y_pred, columns = ['Prediction'])
print(mean_squared_error(y_true = Y_test, y_pred = Y_prediction))
te = te + mean_squared_error(y_true = Y_test, y_pred = Y_prediction)
print(te, te/6)

KeyboardInterrupt: ignored

In [ ]:
import pickle
# Save the XGBoost regressor model — Mean_BMI target (Experiment 1)
model_pkl_file = "reg_model.pkl"

with open(model_pkl_file, 'wb') as file:
    pickle.dump(reg, file)

In [ ]:
files.download("reg_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
training_data = pd.read_csv(TRAINING_LABEL_CSV)
training_data = training_data[X.columns]
training_data = training_data.dropna(axis=0, how='any')
training_data.reset_index(drop = True, inplace = True)
training_data.shape

(36742, 7)

In [ ]:
features = GEE_FEATURES_CSV
training_data = pd.read_csv(TRAINING_LABEL_CSV)
training_data = training_data["DHSID"]
batch_size = 2000
id = []
for df in pd.read_csv(features,chunksize = batch_size):
  df.reset_index(drop = True, inplace = True)
  for i in range(0,df.shape[0]):
    id.append(df['DHSID'][i])

<ipython-input-3-9c0f5dac24ec>:6: DtypeWarning: Columns (3,4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  for df in pd.read_csv(features,chunksize = batch_size):
<ipython-input-3-9c0f5dac24ec>:6: DtypeWarning: Columns (3,4,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  for df in pd.read_csv(features,chunksize = batch_size):
<ipython-input-3-9c0f5dac24ec>:6: DtypeWarning: Columns (3,4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  for df in pd.read_csv(features,chunksize = batch_size):
<ipython-input-3-9c0f5dac24ec>:6: DtypeWarning: Columns (3,4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  for df in pd.read_csv(features,chunksize = batch_size):
<ipython-input-3-9c0f5dac24ec>:6: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  for df in pd.read_csv(features,chunksize = batch_size):
<ipython-input-3-9c0f5

In [ ]:
features = GEE_FEATURES_CSV
training_data = pd.read_csv(TRAINING_LABEL_CSV)
training_data = training_data["DHSID"]
batch_size = 1200

c = 0
# training_data = training_data.dropna(axis=0, how='any')
# training_data.reset_index(drop = True, inplace = True)
ID = training_data.tolist()
print(len(ID))

# f = open(TRAIN_FEATURES_CSV, "w")
# f.truncate()
# f.close()
ct = 0
identifier = []
ids = []

for df in pd.read_csv(features, chunksize=batch_size):

  df.reset_index(drop = True, inplace = True)
  train_features = df.copy()

  for i in range(0,df.shape[0]):
    if(df['DHSID'][i] in ID):
      ct = ct + 1
      identifier.append(1)
    else:
      identifier.append(0)
    c = c + 1
    ids.append(df['DHSID'][i])

  print(ct,c)

  # c = c - ct
  # train_features = df[0:ct]
  # ct=0
  # for i in range(0,df.shape[0]):
  #   if training_data[c]==df["DHSID"][i]:
  #     train_features.iloc[ct]=df.iloc[i]
  #     ct = ct + 1
  #     c = c + 1

  # for i in range(0,train_features.shape[0]):
  #   if(train_features['URBAN_RURA'][i] == 'R'):
  #     train_features['URBAN_RURA'][i] = 0
  #   else:
  #     train_features['URBAN_RURA'][i] = 1
  # train_features['URBAN_RURA'] = train_features['URBAN_RURA'].astype(int)

  # count = 0
  # for i in reversed(range(0,train_features.shape[1])):
  #   if(train_features[train_features.columns[i]].dtype == 'O'):
  #     count = count+1
  #     train_features.drop(train_features.columns[i],axis = 1,inplace = True)

  # train_features.to_csv('train_features.csv', index = False)
  # files.download("train_features.csv")
  # break

dframe = pd.DataFrame(identifier, columns = ['identifier'])
dframe['id_names'] = pd.DataFrame(ids)
  # # print(train_features.shape)
with open(TRAIN_FEATURES_CSV, 'a') as f_object:
  writer_object = writer(f_object)
  if((c-batch_size)==0):
    writer_object.writerow(dframe.columns)
  for o in range(0,dframe.shape[0]):
    writer_object.writerow(dframe.iloc[o])
  f_object.close()

99112


<ipython-input-4-cc2de6a67bbf>:19: DtypeWarning: Columns (3,4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  for df in pd.read_csv(features, chunksize=batch_size):


898 1200


<ipython-input-4-cc2de6a67bbf>:19: DtypeWarning: Columns (3,4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  for df in pd.read_csv(features, chunksize=batch_size):


2021 2400


<ipython-input-4-cc2de6a67bbf>:19: DtypeWarning: Columns (3,4,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  for df in pd.read_csv(features, chunksize=batch_size):


... (105 outputs left out)

<ipython-input-4-cc2de6a67bbf>:19: DtypeWarning: Columns (3,4,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  for df in pd.read_csv(features, chunksize=batch_size):


100446 120000


<ipython-input-4-cc2de6a67bbf>:19: DtypeWarning: Columns (3,4,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  for df in pd.read_csv(features, chunksize=batch_size):


101138 120984


In [ ]:
features = GEE_FEATURES_CSV
training_data = pd.read_csv(TRAINING_LABEL_CSV)
connector = pd.read_csv(TRAIN_FEATURES_CSV)
training_data = training_data[["DHSID", "Mean_BMI"]]
batch_size = 2500

reg = xgb.XGBRegressor(n_estimators=2000)
c = 0
ct = 0
training_data = training_data.dropna(axis=0, how='any')
training_data.reset_index(drop = True, inplace = True)

X_testing = np.empty([0,500], int)
Y_testing = np.empty([0,1], int)

name_list = []
for i in range(0,connector.shape[0]):
  if(connector['identifier'][i] == 1):
    name_list.append(connector['id_names'][i])

arr = connector['identifier'].to_numpy()
p = training_data.shape[0]

for train_features in pd.read_csv(features, chunksize=batch_size):

  train_features.reset_index(drop = True, inplace = True)
  train_id = train_features['DHSID'].tolist()
  val_list = []
  # train_features,c,ct = long_calc(df,training_data,c)

  for i in range(0,train_features.shape[0]):
    if(train_features['URBAN_RURA'][i] == 'R'):
      train_features['URBAN_RURA'][i] = 0
    else:
      train_features['URBAN_RURA'][i] = 1
  train_features['URBAN_RURA'] = train_features['URBAN_RURA'].astype(int)

  # count = 0
  # for i in reversed(range(0,train_features.shape[1])):
  #   if(train_features[train_features.columns[i]].dtype == 'O'):
  #     count = count+1
  feature_list = train_features['DHSID'].tolist()
  train_features.drop(['key1','SOURCE','DHSREGNA','DHSID','DHSCC','DATUM','CCFIPS','ADM1SALBNA','ADM1SALBCO','ADM1NAME','ADM1FIPSNA','ADM1FIPS','new_ind'],axis = 1,inplace = True)

  for i in range(train_features.shape[1]):
    mean_value=train_features[train_features.columns[i]].mean()
    train_features[train_features.columns[i]].fillna(value=mean_value, inplace=True)

  for i in range(train_features.shape[1]):
    mean_value=train_features[train_features.columns[i]].mean()
    train_features[train_features.columns[i]].fillna(value=0, inplace=True)

  arr_passed = arr[ct:(ct+batch_size)]
  m=0
  for i in range(0,arr_passed.shape[0]):
    if (arr_passed[i]==0):
      m = m + 1
  print(m,arr_passed.shape[0])

  train_list = training_data['DHSID'].tolist()
  for k in range(0,train_features.shape[0]):
    if (c < training_data.shape[0] and feature_list[k] in name_list):
      if (train_list[c] in name_list):
        val_list.append(training_data['Mean_BMI'][c])
      else:
        arr_passed[k] = 0
    c = c + 1
  Y = pd.DataFrame(val_list)

  train_f = train_features.to_numpy()

  for i in reversed(range(0,train_f.shape[0])):
    if (arr_passed[i]==0):
      train_f = np.delete(train_f,i,0)

  if (c>p):
    train_f = train_f[0:(p-ct)]
  print(c,train_f.shape[0],Y.shape[0])
  ct = ct + batch_size

  X_train, X_test, Y_train, Y_test = model_selection.train_test_split(train_f, Y, test_size=0.15, train_size=0.85)

  pca = PCA(n_components = 500)
  X_train = pca.fit_transform(X_train)
  X_test = pca.transform(X_test)
  # explained_variance = pca.explained_variance_ratio_

  reg.fit(X_train, Y_train, eval_set=[(X_train, Y_train), (X_test, Y_test)], early_stopping_rounds=500, verbose=True)

  X_testing = np.append(X_testing, X_test, axis=0)
  Y_testing = np.append(Y_testing, Y_test, axis=0)
  Y_pred = reg.predict(X_test)
  Y_prediction = pd.DataFrame(Y_pred, columns = ['Prediction'])
  print(mean_squared_error(y_true = Y_test, y_pred = Y_prediction))

  if(c>p):
    break

# Y_pred = reg.predict(X_testing)
# Y_prediction = pd.DataFrame(Y_pred, columns = ['Prediction'])
# mean_squared_error(y_true = Y_testing, y_pred = Y_prediction)

<ipython-input-3-d026fa096594>:24: DtypeWarning: Columns (3,4,5,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  for train_features in pd.read_csv(features, chunksize=batch_size):
<ipython-input-3-d026fa096594>:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_features['URBAN_RURA'][i] = 0
<ipython-input-3-d026fa096594>:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_features['URBAN_RURA'][i] = 1


379 2500
2500 2121 2121
[0]	validation_0-rmse:15.93728	validation_1-rmse:15.87388


/usr/local/lib/python3.10/dist-packages/xgboost/sklearn.py:835: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


[1]	validation_0-rmse:11.27279	validation_1-rmse:11.22005
[2]	validation_0-rmse:8.03264	validation_1-rmse:7.97847
[3]	validation_0-rmse:5.79330	validation_1-rmse:5.79536
[4]	validation_0-rmse:4.24033	validation_1-rmse:4.29737
[5]	validation_0-rmse:3.18237	validation_1-rmse:3.32724
[6]	validation_0-rmse:2.46202	validation_1-rmse:2.74678
[7]	validation_0-rmse:1.96799	validation_1-rmse:2.43278
[8]	validation_0-rmse:1.63580	validation_1-rmse:2.27690
[9]	validation_0-rmse:1.40240	validation_1-rmse:2.19161
[10]	validation_0-rmse:1.25699	validation_1-rmse:2.14085
[11]	validation_0-rmse:1.14158	validation_1-rmse:2.12357
[12]	validation_0-rmse:1.04855	validation_1-rmse:2.11701
[13]	validation_0-rmse:1.00201	validation_1-rmse:2.11421
[14]	validation_0-rmse:0.93124	validation_1-rmse:2.11564
[15]	validation_0-rmse:0.89172	validation_1-rmse:2.11331
[16]	validation_0-rmse:0.81509	validation_1-rmse:2.11125
[17]	validation_0-rmse:0.77939	validation_1-rmse:2.10834
[18]	validation_0-rmse:0.71993	validat

<ipython-input-3-d026fa096594>:24: DtypeWarning: Columns (3,4,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  for train_features in pd.read_csv(features, chunksize=batch_size):
<ipython-input-3-d026fa096594>:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_features['URBAN_RURA'][i] = 0
<ipython-input-3-d026fa096594>:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_features['URBAN_RURA'][i] = 1


533 2500
5000 1940 1940
[0]	validation_0-rmse:16.01772	validation_1-rmse:15.69998


... (122 outputs left out)

<ipython-input-3-d026fa096594>:24: DtypeWarning: Columns (3,4,5,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  for train_features in pd.read_csv(features, chunksize=batch_size):
<ipython-input-3-d026fa096594>:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_features['URBAN_RURA'][i] = 1
<ipython-input-3-d026fa096594>:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_features['URBAN_RURA'][i] = 0


289 2500
82500 595 595
[0]	validation_0-rmse:16.64430	validation_1-rmse:16.62570
[1]	validation_0-rmse:11.73934	validation_1-rmse:11.71539
[2]	validation_0-rmse:8.33537	validation_1-rmse:8.30401
[3]	validation_0-rmse:5.98058	validation_1-rmse:6.11957
[4]	validation_0-rmse:4.36115	validation_1-rmse:4.63243


/usr/local/lib/python3.10/dist-packages/xgboost/sklearn.py:835: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


[5]	validation_0-rmse:3.23095	validation_1-rmse:3.58948
[6]	validation_0-rmse:2.40329	validation_1-rmse:2.94431
[7]	validation_0-rmse:1.83946	validation_1-rmse:2.52652
[8]	validation_0-rmse:1.44421	validation_1-rmse:2.28115
[9]	validation_0-rmse:1.14024	validation_1-rmse:2.10355
[10]	validation_0-rmse:0.92829	validation_1-rmse:2.05751
[11]	validation_0-rmse:0.79215	validation_1-rmse:2.00276
[12]	validation_0-rmse:0.70014	validation_1-rmse:1.97022
[13]	validation_0-rmse:0.62006	validation_1-rmse:1.94845
[14]	validation_0-rmse:0.56297	validation_1-rmse:1.94147
[15]	validation_0-rmse:0.48985	validation_1-rmse:1.93983
[16]	validation_0-rmse:0.41964	validation_1-rmse:1.92840
[17]	validation_0-rmse:0.37089	validation_1-rmse:1.92383
[18]	validation_0-rmse:0.33524	validation_1-rmse:1.91439
[19]	validation_0-rmse:0.31295	validation_1-rmse:1.91630
[20]	validation_0-rmse:0.28045	validation_1-rmse:1.91465
[21]	validation_0-rmse:0.24119	validation_1-rmse:1.91472
[22]	validation_0-rmse:0.22629	valid

In [ ]:
Y_pred = reg.predict(X_testing)
Y_prediction = pd.DataFrame(Y_pred, columns = ['Prediction'])
mean_squared_error(y_true = Y_testing, y_pred = Y_prediction)

12.932476301031235

In [ ]:
df1 = pd.DataFrame(columns=df.columns)
df1 = df1.append(pd.DataFrame(np.zeros([5000,0])))

<ipython-input-41-872b350d20bd>:2: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df1 = df1.append(pd.DataFrame(np.zeros([5000,0])))


In [ ]:
df1.iloc[379]

new_ind                                   NaN
index                                     NaN
ADM1DHS                                   NaN
ADM1FIPS                                  NaN
ADM1FIPSNA                                NaN
                                         ... 
motorized_only_friction_surface_100000    NaN
motorized_only_travel_time_100000         NaN
key1                                      NaN
key2                                      NaN
key3                                      NaN
Name: 379, Length: 11945, dtype: object

In [ ]:
features = GEE_FEATURES_CSV
submission_file = pd.read_csv(SAMPLE_SUBMISSION_CSV)
submission = submission_file[["DHSID", "Mean_BMI"]]
batch_size = 30000

c = 0
for df in pd.read_csv(features, chunksize=1):
  train_f = pd.DataFrame(columns=df.columns)
  break

name_list = submission['DHSID'].to_numpy()

t=0
a=0
for df in pd.read_csv(features, chunksize=batch_size):
  if(t>a):
    a = t+1

  df.reset_index(drop = True, inplace = True)
  train_id = df['DHSID'].tolist()
  val_list = []

  ct=0
  for i in range(a,len(name_list)):
    if (name_list[i] in train_id):
      ct = ct + 1
      c = c + 1
      t = i

  c = c - ct
  train_features = df1[0:(t-a+1)]
  ct=0
  for i in range(0,len(train_id)):
    if (train_id[i] in name_list):
      train_features.iloc[ct]=df.iloc[i]
      ct = ct + 1
      c = c + 1
    elif(name_list[c] not in train_id):
      # train_features.iloc[ct]=df1.iloc[0]
      ct = ct + 1
      c = c + 1
    if(c == t-a+1):
      break

  for i in range(0,train_features.shape[0]):
    if(train_features['URBAN_RURA'][i] == 'R'):
      train_features['URBAN_RURA'][i] = 0
    else:
      train_features['URBAN_RURA'][i] = 1
  train_features['URBAN_RURA'] = train_features['URBAN_RURA'].astype(int)

  feature_list = train_features['DHSID'].tolist()
  train_features.drop(['key1','SOURCE','DHSREGNA','DHSID','DHSCC','DATUM','CCFIPS','ADM1SALBNA','ADM1SALBCO','ADM1NAME','ADM1FIPSNA','ADM1FIPS','new_ind'],axis = 1,inplace = True)

  # for i in range(train_features.shape[1]):
  #   mean_value=train_features[train_features.columns[i]].mean()
  #   train_features[train_features.columns[i]].fillna(value=mean_value, inplace=True)

  # for i in range(train_features.shape[1]):
  #   mean_value=train_features[train_features.columns[i]].mean()
  #   train_features[train_features.columns[i]].fillna(value=0, inplace=True)

  train_f = train_f.append(train_features)
  print(ct,c,t)
print(train_f.shape[0])
train_f.head()

#   pca = PCA(n_components = 500)
#   X_train = pca.fit_transform(X_train)
#   X_test = pca.fit_transform(X_test)
#   # explained_variance = pca.explained_variance_ratio_

#   X_testing = np.append(X_testing, X_test, axis=0)
#   Y_testing = np.append(Y_testing, Y_test, axis=0)
#   Y_pred = reg.predict(X_test)
#   Y_prediction = pd.DataFrame(Y_pred, columns = ['Prediction'])
#   print(mean_squared_error(y_true = Y_test, y_pred = Y_prediction))

# Y_pred = reg.predict(X_testing)
# Y_prediction = pd.DataFrame(Y_pred, columns = ['Prediction'])
# mean_squared_error(y_true = Y_testing, y_pred = Y_prediction)

<ipython-input-42-77ab4dda8c91>:15: DtypeWarning: Columns (3,4,5,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  for df in pd.read_csv(features, chunksize=batch_size):


In [ ]:
print(train_features.shape)
df1.shape

(379, 11945)


(1, 11945)

In [ ]:
train_features.iloc[ct]=df1.iloc[0]

IndexError: ignored

In [ ]:
train_f.to_csv('sample_features.csv', index = False)
files.download("sample_features.csv")

KeyboardInterrupt: ignored

In [ ]:
df1.shape

(1, 11945)

In [ ]:
count = 0
for i in range(0,len(name_list)):
  if(name_list[i] in all):
    count = count +1

In [ ]:
count

14975

In [ ]:
a = [1,2,np.NaN]
print(a[0])
print(a[2])

1
nan


In [ ]:
s_features = pd.read_csv(SAMPLE_FEATURES_CSV)
X_S = pca.transform(s_features)
Y = model.predict


Y_pred.to_csv('feature3_pred.csv', index = False)
files.download("feature3_pred.csv")